In [21]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score

from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.ensemble import VotingClassifier

from tensorflow import keras
from keras import layers

In [2]:
df = pd.read_csv('Titanic-Dataset.csv')
print(df.head())
print(df.info())

   PassengerId  Survived  Pclass  \
0            1         0       3   
1            2         1       1   
2            3         1       3   
3            4         1       1   
4            5         0       3   

                                                Name     Sex   Age  SibSp  \
0                            Braund, Mr. Owen Harris    male  22.0      1   
1  Cumings, Mrs. John Bradley (Florence Briggs Th...  female  38.0      1   
2                             Heikkinen, Miss. Laina  female  26.0      0   
3       Futrelle, Mrs. Jacques Heath (Lily May Peel)  female  35.0      1   
4                           Allen, Mr. William Henry    male  35.0      0   

   Parch            Ticket     Fare Cabin Embarked  
0      0         A/5 21171   7.2500   NaN        S  
1      0          PC 17599  71.2833   C85        C  
2      0  STON/O2. 3101282   7.9250   NaN        S  
3      0            113803  53.1000  C123        S  
4      0            373450   8.0500   NaN        S  
<c

In [14]:
print(df.isnull().sum())

Survived      0
Pclass        0
Sex           0
Age           0
SibSp         0
Parch         0
Fare          0
Embarked_Q    0
Embarked_S    0
dtype: int64


In [4]:
#เติม age ด้วย median(ค่ากลาง)
df['Age'] = df['Age'].fillna(df['Age'].median())

#เติม Embarked ด้วย mode(ซ้ำเยอะสุด)
df['Embarked'] = df['Embarked'].fillna(df['Embarked'].mode()[0])

#Cabin มี null เยอะมาก ลบทิ้ง
df.drop('Cabin', axis=1, inplace=True)

#ลบ column ทิ้งเพราะเป็นค่า unique ไม่ได้จำเป็นต่อ model
df.drop(['PassengerId','Name','Ticket'], axis=1, inplace=True)

In [5]:
#แปลงค่าเป็นตัวเลข
df['Sex'] = df['Sex'].map( {'male':0,'female':1} )
df = pd.get_dummies( df, columns=['Embarked'], drop_first=True )

In [6]:
#กำหนด target
X = df.drop('Survived', axis=1)
y = df['Survived']

In [7]:
X_train, X_test, y_train, y_test = train_test_split( X, y, test_size=0.2, random_state=42 )
scaler = StandardScaler()
X_train = scaler.fit_transform( X_train )
X_test = scaler.transform( X_test )

In [8]:
#Machine Learning แบบ ensemble

#random forest
rf = RandomForestClassifier(n_estimators=100)

#KNN
knn = KNeighborsClassifier(n_neighbors=15)

#SVM
svm = SVC(probability=True)

In [9]:
#รวม model
ensemble = VotingClassifier( estimators=[('rf',rf), ('knn',knn), ('svm',svm)], voting='soft' )

In [10]:
ensemble.fit( X_train, y_train )
pred = ensemble.predict (X_test )
accuracy = accuracy_score( y_test, pred )
print( "Ensemble Accuracy =", accuracy )

Ensemble Accuracy = 0.8156424581005587


In [22]:
model = keras.Sequential()
model.add(keras.Input(shape=(X_train.shape[1],)))

model.add(layers.Dense(128, activation='relu'))
model.add(layers.BatchNormalization())
model.add(layers.Dropout(0.3))

model.add(layers.Dense(64, activation='relu'))
model.add(layers.BatchNormalization())
model.add(layers.Dropout(0.3))

model.add(layers.Dense(32, activation='relu'))

model.add(layers.Dense(1, activation='sigmoid'))

In [12]:
model.compile( optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'] )

In [13]:
history = model.fit( X_train, y_train, epochs=80, batch_size=32, validation_split=0.2 )
loss, accuracy = model.evaluate(X_test, y_test)
print("Neural Network Accuracy =", accuracy)

Epoch 1/80
18/18 ━━━━━━━━━━━━━━━━━━━━ 4s 30ms/step - accuracy: 0.6678 - loss: 0.6394 - val_accuracy: 0.7972 - val_loss: 0.5956
Epoch 2/80
18/18 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - accuracy: 0.7627 - loss: 0.5063 - val_accuracy: 0.8112 - val_loss: 0.5600
Epoch 3/80
18/18 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - accuracy: 0.8049 - loss: 0.4858 - val_accuracy: 0.8112 - val_loss: 0.5495
Epoch 4/80
18/18 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step - accuracy: 0.7909 - loss: 0.4821 - val_accuracy: 0.7902 - val_loss: 0.5377
Epoch 5/80
18/18 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - accuracy: 0.7944 - loss: 0.4787 - val_accuracy: 0.8112 - val_loss: 0.5243
Epoch 6/80
18/18 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.7944 - loss: 0.4607 - val_accuracy: 0.8252 - val_loss: 0.5081
Epoch 7/80
18/18 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.7926 - loss: 0.4737 - val_accuracy: 0.7972 - val_loss: 0.5033
Epoch 8/80
18/18 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.7909 - loss: 0.4563 - val_accuracy: 0.7972 - val_l